In [1]:
import xarray as xr 
import gcsfs
from dask.diagnostics import ProgressBar
from pathlib import Path
import numpy as np


In [2]:
aifs_ens_v2 = "/net/monsoon/marchakitus/AIFS/v2p0/combined/forecasts_AIFS_ENS_v2"

fs = gcsfs.GCSFileSystem(token="anon")
era5_store = fs.get_mapper(
    "weatherbench2/datasets/era5/"
    "1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr"
)


In [3]:
paths = sorted(Path(aifs_ens_v2).glob("*.zarr"))
wanted = ['2d', '2t']

with ProgressBar():
    ds = xr.open_mfdataset(
        paths,
        engine="zarr",
        combine="nested",
        concat_dim="time",
        preprocess=lambda x: x[wanted],
        chunks={
            "time": 1,       # unavoidable: one time per store
            "number": 26,    # combine all ensemble members
            "step": 24,
            "latitude": 180,
            "longitude": 180,
        },        
        parallel=True,
        data_vars="all",
        coords="minimal",
        compat="override",
        join="override",
        combine_attrs="override",
        consolidated=None,
    )

ds = ds.rename({'2d': '2m_dewpoint_temperature', '2t': '2m_temperature', 'lat': 'latitude', 'lon': 'longitude' })

[########################################] | 100% Completed | 131.53 s


In [4]:
era5 = xr.open_zarr(
    era5_store,
    chunks={},
    consolidated=True,
)

In [5]:
times = ds.indexes["time"].intersection(era5.indexes["time"])
era5_temps = era5[["2m_dewpoint_temperature", "2m_temperature"]].sel(time=times)
ds_temps = ds.sel(time=times)

### RMSE

In [6]:
squared_residual = (era5_temps - ds_temps)**2
rmse = np.sqrt( squared_residual.mean('time') )

### Probability of Exceedance

In [18]:
def probability_of_exceedance(da, threshold=32, member_dim='number'):
    """ returns probability of exceeding given threshold (can be DataArray or int/float) by member counting """
    return (da > threshold).mean(member_dim)

def observation_did_exceed(obs, threshold=32):
    return (obs > threshold)

def hit_rate(forecast_event, observed_event, dim=None):
    """Fraction of observed events correctly forecast: TP / (TP + FN)."""
    hits = forecast_event & observed_event
    return hits.sum(dim) / observed_event.sum(dim)

def miss_rate(forecast_event, observed_event, dim=None):
    """Fraction of observed events missed: FN / (TP + FN)."""
    misses = ~forecast_event & observed_event
    return misses.sum(dim) / observed_event.sum(dim)

def false_positive_rate(forecast_event, observed_event, dim=None):
    """Fraction of observed non-events incorrectly forecast: FP / (FP + TN)."""
    false_positives = forecast_event & ~observed_event
    return false_positives.sum(dim) / (~observed_event).sum(dim)


In [16]:
getattr(rmse, '2m_dewpoint_temperature').sizes

Frozen({'latitude': 721, 'longitude': 1440, 'number': 26, 'prediction_timedelta': 200})

In [21]:
ds_temps['2m_temperature'].groupby("prediction_timedelta.days").mean()

<xarray.DataArray '2m_temperature' (time: 2096, number: 26, days: 51,
                                    latitude: 721, longitude: 1440)> Size: 12TB
dask.array<transpose, shape=(2096, 26, 51, 721, 1440), dtype=float32, chunksize=(1, 26, 1, 90, 180), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 17kB 2000-01-01 2000-01-07 ... 2023-01-09
  * number     (number) int64 208B 0 1 2 3 4 5 6 7 8 ... 18 19 20 21 22 23 24 25
  * days       (days) int64 408B 0 1 2 3 4 5 6 7 8 ... 43 44 45 46 47 48 49 50
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8